In [1]:
%%writefile app.py

import streamlit as st
import pandas as pd
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

st.title("📰 Fake News Detection")

st.write("Enter a news article below to check whether it is Fake or Real.")

fake = pd.read_csv(r"C:\Users\Hp\Downloads\Fake.csv.zip")
true = pd.read_csv(r"C:\Users\Hp\Downloads\True.csv.zip")

fake["label"] = 0
true["label"] = 1

data = pd.concat([fake, true], ignore_index=True)

data = data.sample(frac=1, random_state=42).reset_index(drop=True)


def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\d+", "", text)

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


data["content"] = data["title"] + " " + data["text"]
data["content"] = data["content"].apply(clean_text)

X = data["content"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.7
)

X_train = vectorizer.fit_transform(X_train)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


news = st.text_area(
    "Enter News Article:",
    height=250
)

if st.button("Predict"):

    if news.strip() == "":
        st.warning("Please enter a news article.")

    else:
        cleaned_news = clean_text(news)

        news_vector = vectorizer.transform([cleaned_news])

        prediction = model.predict(news_vector)

        if prediction[0] == 1:
            st.success("✅ Real News")
        else:
            st.error("❌ Fake News")

Writing app.py


In [1]:
import subprocess

subprocess.Popen(["streamlit", "run", "app.py"])

<Popen: returncode: None args: ['streamlit', 'run', 'app.py']>